# WtDtCore 架构

## 概述

WtDtCore（数据传输核心）是WonderTrader框架中负责行情数据接收、处理、存储和分发的核心模块。该模块采用分层架构设计，实现了高性能、高可靠性的实时数据处理系统。
```mermaid
graph TB
    subgraph "第1层：接口定义层"
        IDataCaster[IDataCaster.h<br/>广播器接口]
    end
    
    subgraph "第2层：核心管理层"
        DataManager[DataManager.h/.cpp<br/>数据管理中枢]
        StateMonitor[StateMonitor.h/.cpp<br/>状态监控器]
        ParserAdapter[ParserAdapter.h/.cpp<br/>解析器适配器]
    end
    
    subgraph "第3层：功能实现层"
        UDPCaster[UDPCaster.h/.cpp<br/>UDP广播实现]
        ShmCaster[ShmCaster.h/.cpp<br/>共享内存广播实现]
        IndexFactory[IndexFactory.h/.cpp<br/>指数工厂]
        IndexWorker[IndexWorker.h/.cpp<br/>指数计算器]
    end
    
    subgraph "第4层：辅助工具层"
        WtHelper[WtHelper.h/.cpp<br/>路径工具]
        StatHelper[StatHelper.hpp<br/>统计工具]
    end
    
    %% 依赖关系
    DataManager --> IDataCaster
    UDPCaster --> IDataCaster
    ShmCaster --> IDataCaster
    ParserAdapter --> DataManager
    StateMonitor --> DataManager
    IndexFactory --> DataManager
    IndexWorker --> IndexFactory
    DataManager --> WtHelper
    UDPCaster --> StatHelper
    ShmCaster --> StatHelper
```

## 核心文件详细分析

### 1. 数据管理中枢 - DataManager

```mermaid
graph LR
    subgraph "DataManager 核心职责"
        A[行情数据接收] --> B[数据验证与过滤]
        B --> C[存储引擎调度]
        C --> D[广播器管理]
        D --> E[状态控制集成]
    end
    
    subgraph "接口实现"
        F[IDataWriterSink<br/>为Writer提供回调]
    end
    
    subgraph "设计模式"
        G[外观模式<br/>Facade Pattern]
        H[适配器模式<br/>Adapter Pattern]
        I[策略模式<br/>Strategy Pattern]
    end
    
    A -.-> F
    C -.-> G
    F -.-> H
    D -.-> I
```

**核心特点：**
- **中枢角色**：作为数据流转的核心枢纽，协调Parser、Writer、Caster、StateMonitor
- **双重身份**：既是管理门面，又是Writer的回调接收器
- **动态加载**：支持运行时加载不同的存储引擎（WtDataStorage、自定义Writer）
- **多播支持**：同时支持多个数据广播器并行工作

### 2. 行情解析适配器 - ParserAdapter

```mermaid
graph TB
    subgraph "ParserAdapter 工作流程"
        A[动态加载Parser模块] --> B[解析过滤器配置]
        B --> C[智能订阅策略]
        C --> D[行情数据接收]
        D --> E[数据验证与转发]
    end
    
    subgraph "订阅策略优先级"
        F[1. Code Filter<br/>合约代码过滤]
        G[2. Exchange Filter<br/>交易所过滤]
        H[3. Full Market<br/>全市场订阅]
        F --> G --> H
    end
    
    subgraph "适配器模式应用"
        I[IParserApi<br/>行情解析器接口]
        J[IParserSpi<br/>行情回调接口]
        K[ParserAdapter<br/>适配器实现]
        I --> K
        K --> J
    end
```

**核心特点：**
- **适配器模式**：将不同厂商的Parser统一适配到框架接口
- **智能订阅**：支持品种级、交易所级、全市场级订阅策略
- **动态加载**：运行时加载Parser动态库，支持多种行情源
- **数据过滤**：在接入层就进行数据过滤，提高处理效率

### 3. 状态监控器 - StateMonitor

```mermaid
stateDiagram-v2
    [*] --> SS_ORIGINAL : 系统启动
    SS_ORIGINAL --> SS_INITIALIZED : 到达初始化时间
    SS_INITIALIZED --> SS_RECEIVING : 到达开盘时间
    SS_RECEIVING --> SS_PAUSED : 中途休盘
    SS_PAUSED --> SS_RECEIVING : 恢复交易
    SS_RECEIVING --> SS_CLOSED : 到达收盘时间
    SS_CLOSED --> SS_PROCING : 到达盘后处理时间
    SS_PROCING --> SS_PROCED : 处理完成
    SS_PROCED --> SS_ORIGINAL : 下一交易日
    
    SS_ORIGINAL --> SS_Holiday : 检测到节假日
    SS_INITIALIZED --> SS_Holiday : 检测到节假日
    SS_Holiday --> SS_ORIGINAL : 下一交易日
    
    note right of SS_RECEIVING : 可接收数据状态
    note right of SS_PROCING : 触发历史数据转储
```

**核心特点：**
- **有限状态机**：精确控制数据接收和处理的时机
- **多时段管理**：支持不同交易时段的独立状态管理
- **自动转换**：基于时间和交易日历自动进行状态转换
- **盘后处理**：自动触发历史数据转储和缓存清理

### 4. 数据广播器架构

```mermaid
graph TB
    subgraph "IDataCaster 接口层"
        A[IDataCaster<br/>统一广播接口]
    end
    
    subgraph "具体实现层"
        B[UDPCaster<br/>UDP网络广播]
        C[ShmCaster<br/>共享内存广播]
        D[CustomCaster<br/>自定义广播器]
    end
    
    subgraph "UDPCaster 特性"
        E[单播模式<br/>点对点传输]
        F[广播模式<br/>局域网广播]
        G[组播模式<br/>组播传输]
        H[订阅服务<br/>客户端订阅]
    end
    
    subgraph "ShmCaster 特性"
        I[无锁环形队列<br/>Lock-Free Ring Buffer]
        J[纳秒级延迟<br/>极致性能]
        K[进程间通信<br/>本地IPC]
        L[联合体设计<br/>节省内存]
    end
    
    A --> B
    A --> C
    A --> D
    
    B --> E
    B --> F
    B --> G
    B --> H
    
    C --> I
    C --> J
    C --> K
    C --> L
```

**性能对比：**
- **UDPCaster**：延迟1-10ms，适用于网络分发，支持跨机器
- **ShmCaster**：延迟<100ns，适用于本地进程间通信，性能极致

# 数据广播器接口类 IDataCaster.h
该接口定义了数据广播的标准行为规范。所有需要实现数据分发功能的类都应该继承此接口并实现其虚函数。
已知实现类：
- UDPCaster：UDP网络广播器
- ShmCaster：共享内存广播器
```cpp
class IDataCaster
{
public:
	/**
	 * @brief 广播Tick行情数据（纯虚函数，必须实现）
	 * @param curTick 当前Tick行情数据的指针，不能为NULL（调用者负责检查）
	 */
	virtual void	broadcast(WTSTickData* curTick) = 0;
	
	/**
	 * @brief 广播委托队列数据（有默认实现的虚函数）
	 * 用于广播Level-2行情中的委托队列数据。委托队列显示了最优买卖价位上的前N笔委托明细（通常是50笔）。
	 * @param curOrdQue 当前委托队列数据的指针
	 */
	virtual void	broadcast(WTSOrdQueData* curOrdQue){}
	
	/**
	 * @brief 广播逐笔委托数据（有默认实现的虚函数）
	 * 该方法用于广播Level-2行情中的逐笔委托数据。逐笔委托记录了每一笔委托单的详细信息。
	 * @param curOrdDtl 当前逐笔委托数据的指针
	 */
	virtual void	broadcast(WTSOrdDtlData* curOrdDtl){}
	
	/**
	 * @brief 广播逐笔成交数据（有默认实现的虚函数）
	 * 该方法用于广播Level-2行情中的逐笔成交数据。逐笔成交记录了市场上每一笔真实成交的详细信息。
	 * @param curTrans 当前逐笔成交数据的指针
	 */
	virtual void	broadcast(WTSTransData* curTrans){}
};
```

# 基于UDP协议的网络数据广播模块 UDPCaster.h/.cpp
提供了通过网络（UDP协议）向一个或多个远程客户端广播实时数据的能力。它解决了**跨机器**、**一对多**的数据分发问题。
1.  **分布式系统支持**: 一台高性能的行情接收服务器可以运行 WtDtCore 模块，然后通过 UDPCaster 将处理好的行情数据分发给局域网内的多台策略服务器。
2.  **解耦系统**: 允许策略运行、风险监控、行情展示等不同功能的程序部署在不同的物理机器上，每个程序独立接收行情，互不干扰。
3.  **提供多种广播模式**:
      * **单播**: 点对点精确发送给指定的IP地址。
      * **广播**: 发送到局域网的广播地址（如 255.255.255.255），网内所有主机都能收到。
      * **组播**: 发送到一个特定的组播地址（如 239.1.1.1），只有加入了该组播组的机器才能收到，比广播更高效。
4.  **提供订阅服务**: 允许客户端在任何时候加入数据流，并能请求获取最新的行情快照，以快速同步状态。

## 成员
- `boost::asio::ip::udp::endpoint m_senderEP`：发送者端点（接收订阅请求时记录）
- `char m_data[max_length]`：接收缓冲区
- `UDPSocketPtr	m_sktBroadcast`：广播套接字（用于发送）
- `UDPSocketPtr	m_sktSubscribe`：订阅套接字（用于接收订阅请求）
  - typedef std::shared_ptr\<UDPSocket\> `UDPSocketPtr`：UDP套接字智能指针类型
  - typedef boost::asio::ip::udp::socket `UDPSocket`：UDP套接字类型
- **单播/广播接收者列表（按数据格式分类）**
  - `ReceiverList m_listFlatRecver`：Flat格式接收者列表
  - `ReceiverList m_listJsonRecver`：JSON格式接收者列表
  - `ReceiverList m_listRawRecver`：Raw格式接收者列表
  - 其中
    - typedef std::vector\<UDPReceiverPtr\> `ReceiverList`：接收者列表类型
    - typedef std::shared_ptr\<UDPReceiver\> `UDPReceiverPtr`;
    - UDPReceiver
		```cpp
		/* UDP接收者结构，存储单个UDP接收者的信息 */
		typedef struct tagUDPReceiver
		{
			EndPoint _ep;	// 端点（IP地址+端口）
			uint32_t _type;	// 数据格式类型（0=Flat, 1=JSON, 2=Raw）
		} UDPReceiver;
		```
- **组播接收者列表（按数据格式分类，每个有独立的Socket）**
  - `MulticastList m_listFlatGroup`：Flat格式组播列表
  - `MulticastList m_listJsonGroup`：JSON格式组播列表
  - `MulticastList m_listRawGroup`：Raw格式组播列表
    - typedef std::vector\<MulticastPair\> `MulticastList`：组播列表类型
    - typedef std::pair\<UDPSocketPtr,UDPReceiverPtr\> `MulticastPair`：组播对（Socket + 接收者信息）

- `boost::asio::io_service m_ioservice`：Boost.Asio IO服务对象
- `StdThreadPtr	m_thrdIO`：IO线程（运行io_service）

- `StdThreadPtr	m_thrdCast`：广播线程（处理数据队列）
- `StdCondVariable m_condCast`：广播线程条件变量
- `StdUniqueMutex m_mtxCast`：广播线程互斥锁
- `bool m_bTerminated`：终止标志

- `WTSBaseDataMgr* m_bdMgr`：基础数据管理器指针
- `DataManager*	m_dtMgr`：数据管理器指针

- `std::queue<CastData>	m_dataQue`：数据队列（缓冲待广播的数据）
	```cpp
	/* 广播数据包装结构（带引用计数）*/
	typedef struct _CastData
	{
		uint32_t _datatype;	// 数据类型（消息类型）
		WTSObject* _data;	// 数据对象指针（基类指针）
	} CastData;
	```

## 方法

### 初始化与生命周期

#### 初始化 init
```cpp
/**
 * @brief 初始化UDP广播器实现。从配置中读取广播和组播接收者列表，初始化UDP服务。
 * 
 * @param cfg 配置参数对象
 * @param bdMgr 基础数据管理器指针
 * @param dtMgr 数据管理器指针
 * @return bool 初始化成功返回true，失败返回false
 */
bool UDPCaster::init(WTSVariant* cfg, WTSBaseDataMgr* bdMgr, DataManager* dtMgr)
```

参数 cfg 的例子：
```json
{
    "active": true,
    "sport": 3997,
    "broadcast": [
        { 
            "host": "192.168.1.101", 
            "port": 9001, 
            "type": 2
        },
        { 
            "host": "192.168.1.102", 
            "port": 9001, 
            "type": 2
        },
        { 
            "host": "192.168.1.255", 
            "port": 9002, 
            "type": 2 
        }
    ],
    "multicast": [
        { 
            "host": "239.1.1.1", 
            "port": 9003, 
            "sendport": 0,
            "type": 2 
        }
    ]
}
```
- `active`：是否激活此 UDPCaster 实例
- `sport`：服务端口。此 UDPCaster 实例会监听本机的该端口：当一个客户端（如策略程序、监控UI）向这个端口发送请求时，UDPCaster 可以接收到
- `broadcast`：定义点对点（单播）和局域网广播的目标
  - 单播，例如 `{ "host": "192.168.1.101", "port": 9001, "type": 2 }`
    - 会将每一条行情数据都单独地、点对点地发送到 IP 地址为 192.168.1.101 和 192.168.1.102 的机器上的 9001 端口。
  - 广播，例如 `{ "host": "192.168.1.255", "port": 9002, "type": 2 }`
    - 会将每一条行情数据发送到 192.168.1.255 这个广播地址。网络交换机会负责将这个数据包复制并发送给该子网（192.168.1.0/24）内的所有设备
  - `type`：指定数据包的格式。0=Flat, 1=JSON, 2=Raw
- `multicast`：用于定义组播分发的目标。这是一种介于单播和广播之间的更高效的广播方式
  - 例如，`{ "host": "239.1.1.1", "port": 9003, "sendport": 0, "type": 2 }`。会将每一条行情数据发送到 239.1.1.1 这个组播地址的 9003 端口
  - 组播数据包只会被网络交换机转发给那些明确声明“加入”了 239.1.1.1 这个组播组的设备
  - `sendport`：指定本地发送端口。0 表示由操作系统自动选择一个可用的端口进行发送

#### 启动服务 start

#### 停止服务 stop

### 接收者管理

#### 添加广播/单播接收者 addBRecver

#### 添加组播接收者 addMRecver

### IDataCaster 接口实现

#### 广播Tick数据 broadcast

#### 广播委托队列数据 broadcast

#### 广播逐笔委托数据 broadcast

#### 广播逐笔成交数据 broadcast

### 内部异步与核心逻辑

#### 处理广播发送回调 handle_send_broad

#### 处理组播发送回调 handle_send_multi

#### 异步接收订阅请求 do_receive

#### 执行数据广播（内部实现） do_broadcast

# 共享内存数据广播器 ShmCaster.h/cpp
提供了基于共享内存的、纳秒级延迟的进程间通信能力，是 WonderTrader 中最快的**单机**数据广播方式。

## 成员
- `std::string _path`：共享内存文件路径。生产者和消费者必须使用相同的路径
- `MappedFilePtr _mapfile`：内存映射文件智能指针
  - typedef std::shared_ptr\<BoostMappingFile\> MappedFilePtr;
- `CastQueue* _queue`：队列结构指针，指向共享内存中的队列结构，多个进程中的该指针指向同一块物理内存
  - typedef `_DataQueue`\<8*1024\> `CastQueue`：广播队列类型
    ```cpp
    template <int N = 8*1024>               // 模板参数N：队列容量，默认8192
    struct _DataQueue
    {
      uint64_t	_capacity = N;              // 队列容量（编译期常量，默认8192）
      volatile uint64_t	_readable;          // 可读位置指针（消费者读取，生产者更新），volatile确保多进程可见
      volatile uint64_t	_writable;          // 可写位置指针（生产者更新），volatile确保多进程可见
      uint32_t	_pid;                       // 生产者进程ID（用于调试和监控）
      DataItem	_items[N];                  // 数据项数组（环形缓冲区）
    };

    typedef struct _DataItem                // 数据项结构（支持多种数据类型）
    {
      uint32_t	_type;	                    // 数据类型标识：0-tick, 1-委托队列, 2-逐笔委托, 3-逐笔成交
      
      /* 联合体：同一内存位置存储不同类型的数据，任一时刻只有一个成员有效，由_type决定 */
      union
      {
        WTSTickStruct	_tick;                // Tick行情数据（_type=0时有效）
        WTSOrdQueStruct _queue;             // 委托队列数据（_type=1时有效）
        WTSOrdDtlStruct	_order;             // 逐笔委托数据（_type=2时有效）
        WTSTransStruct	_trans;             // 逐笔成交数据（_type=3时有效）
      };
    } DataItem;
    ```
- `bool _inited`：初始化标志

## 方法

### 初始化 init

### IDataCaster 接口实现

#### 广播Tick数据 broadcast

#### 广播委托队列数据 broadcast

#### 广播逐笔委托数据 broadcast

#### 广播逐笔成交数据 broadcast

# 数据传输统计 StatHelper.hpp
提供了一个全局唯一的、线程安全的统计信息中心：对系统关键路径（主要是数据广播）的性能和状态进行量化度量。

## 源码
```cpp
class StatHelper
{
public:
	/* 获取StatHelper单例实例 */
	static StatHelper& one()
	{
		static StatHelper only;
		return only;
	}

public:
	/* 统计信息数据结构。存储某一类型的统计数据，包括接收包数、发送包数和发送字节数 */
	typedef struct _StatInfo
	{
		uint32_t	_recv_packs;        // 接收的数据包数量（4字节）
		uint32_t	_send_packs;        // 发送的数据包数量（4字节）
		uint64_t	_send_bytes;        // 发送的字节总数（8字节）

		_StatInfo()
		{
			_recv_packs = 0;
			_send_bytes = 0;
			_send_packs = 0;
		}
	} StatInfo;

	/* 统计类型枚举。定义了不同的统计类型，每种类型维护独立的统计信息。
	 * 当前只定义了广播统计，未来可扩展更多类型：
	 * - 可以添加ST_PARSER（解析器统计）
	 * - 可以添加ST_TRADER（交易通道统计）
	 * - 可以添加ST_ENGINE（引擎统计）
	 */
	typedef enum
	{
		ST_BROADCAST	// 广播统计类型（UDP、共享内存等数据分发）
	} StatType;

public:
	/**
	 * @brief 更新统计信息（原子操作，线程安全）
	 * @param sType 统计类型（如ST_BROADCAST）
	 * @param recvPacks 本次接收的包数（增量值）
	 * @param sendPacks 本次发送的包数（增量值）
	 * @param sendBytes 本次发送的字节数（增量值）
	 */
	void updateStatInfo(StatType sType, uint32_t recvPacks, uint32_t sendPacks, uint64_t sendBytes)
	{
		// 获取写锁，独占访问统计数据
		// BoostWriteLock是RAII类，析构时自动释放锁
		BoostWriteLock lock(_mutexes[sType]);
		
		StatInfo& sInfo = _stats[sType];
		sInfo._recv_packs += recvPacks;
		sInfo._send_packs += sendPacks;
		
		// 检测sendBytes是否会溢出
		// 如果：UINT64_MAX - 当前值 < 要增加的值
		// 则：当前值 + 要增加的值 会超过UINT64_MAX
		if(UINT64_MAX - sInfo._send_bytes < sendBytes)
			// 溢出：重置为新值（而不是累加）
			// 这会导致统计数据不连续，但避免了溢出
			sInfo._send_bytes = sendBytes;
		else
			// 正常情况：累加字节数
			sInfo._send_bytes += sendBytes;
	}
	
	/**
	 * @brief 读取指定类型的当前统计信息（支持多线程并发读取）
	 * @param sType 统计类型（如ST_BROADCAST）
	 * @return StatInfo 统计信息的副本
	 */
	StatInfo getStatInfo(StatType sType)
	{
		BoostReadLock lock(_mutexes[sType]);
		return _stats[sType];
	}

private:
	// 存储不同类型的统计信息，数组大小为5，支持5种统计类型。
	StatInfo		_stats[5];
	// 读写锁数组，为每种统计类型提供独立的读写锁
	// 多个读者可以同时持有锁；写者独占锁，阻塞所有其他读者和写者
	BoostRWMutex	_mutexes[5];	
};
```

# 路径管理 WtHelper.h/.cpp

## 源码
```cpp
class WtHelper
{
public:
    /* 获取当前工作目录实现 */
	static const char* get_cwd()
    {
        static std::string _cwd;
        if(_cwd.empty())
        {
            char   buffer[255];
    #ifdef _MSC_VER
            _getcwd(buffer, 255);
    #else	//UNIX
            getcwd(buffer, 255);
    #endif
            _cwd = buffer;
            _cwd = StrUtil::standardisePath(_cwd);
        }	
        return _cwd.c_str();
    }

    /* 设置 WonderTrader 框架模块所在的目录路径。通常在程序启动初始化阶段调用 */
	static void set_module_dir(const char* mod_dir){ _bin_dir = mod_dir; }
	static const char* get_module_dir(){ return _bin_dir.c_str(); }
private:
	static std::string	_bin_dir;   // WonderTrader 模块目录路径静态存储
};
```

# 指数工厂类 IndexFactory.h/cpp
负责管理和协调多个指数工作器。

## 成员
- `IndexWorkers _workers`：指数工作器，存储所有的 IndexWorker 实例，每个负责一个指数的计算
  - typedef std::vector\<IndexWorkerPtr\>	IndexWorkers;
  - typedef std::shared_ptr\<IndexWorker\> IndexWorkerPtr;
- `IHotMgr* _hot_mgr`：主力合约管理器指针
- `IBaseDataMgr* _bd_mgr`：基础数据管理器指针
- `DataManager*	_data_mgr`：数据管理器指针
- `ThreadPoolPtr _pool`：线程池智能指针，IndexWorker的计算任务在线程池中并行执行
  - typedef std::shared_ptr\<boost::threadpool::pool\> ThreadPoolPtr;
- `wt_hashset<std::string> _subbed`：已订阅合约集合，存储所有被Worker订阅的合约代码

## 方法

### 初始化 init
初始化指数工厂：从配置中加载所有指数定义，为每个活跃的指数创建 IndexWorker。

流程：
- 将 hotMgr、bdMgr、dataMgr 传递给 `_hot_mgr`、`_bd_mgr`、`_data_mgr`
- 从 config 中
  - 读取 `poolsize`，如果大于零则以读取的大小创建线程池 `_pool`
  - 读取指数配置 `indice`，遍历每个配置
    - 如果 `active` 项为 false，跳过该配置
    - 否则：将 this 传入来创建一个新的 IndexWorker 并用当前配置初始化，然后将创建好的 IndexWorker 添加到 `_worker`
```cpp
/* @param config 配置参数对象
 * @param hotMgr 主力合约管理器指针
 * @param bdMgr 基础数据管理器指针
 * @param dataMgr 数据管理器指针
 * @return bool 初始化成功返回true，失败返回false
 */
bool IndexFactory::init(WTSVariant* config, IHotMgr* hotMgr, IBaseDataMgr* bdMgr, DataManager* dataMgr)
```
配置 config 例子：
```json
{
    "poolsize": 4,
    "indice": [
        {
            "active": true,
            "exchg": "WTS",
            "code": "BLK",
            "name": "黑色系主力合约指数",
            "trigger": "SHFE.rb.HOT",
            "timeout": 100,
            "weight_alg": 1,
            "stand_scale": 1.0,
            "codes": [
                { "code": "SHFE.rb.HOT", "weight": 0.4 },
                { "code": "DCE.i.HOT", "weight": 0.4 },
                { "code": "DCE.j.HOT", "weight": 0.2 }
            ]
        },
        {
            "active": true,
            "exchg": "WTS",
            "code": "AGRI",
            "name": "农产品活跃度指数",
            "trigger": "time",
            "timeout": 0,
            "weight_alg": 2,
            "stand_scale": 1000,
            "commodities": [
                "CZCE.CF",
                "DCE.m",
                "DCE.p",
                "DCE.y"
            ]
        },
        {
            "active": true,
            "exchg": "WTS",
            "code": "FIN",
            "name": "金融期货加权指数",
            "trigger": "time",
            "timeout": 50,
            "weight_alg": 0,
            "stand_scale": 1.0,
            "codes": [
                "CFFEX.IF.HOT",
                "CFFEX.IH.HOT",
                "CFFEX.IC.HOT"
            ]
        },
        {
            "active": false,
            "exchg": "WTS",
            "code": "TEST",
            "name": "测试指数(未激活)",
            "trigger": "time",
            "timeout": 0,
            "weight_alg": 0,
            "codes": ["SHFE.au.HOT"]
        }
    ]
}
```

### 订阅成分合约的Tick数据 sub_ticks
IndexWorker 在初始化时调用此方法订阅成分合约，并获取当前行情作为基准
```cpp
/* @param fullCode 完整合约代码（如"SHFE.rb2105"）
 * @return WTSTickData* 该合约当前的Tick数据（可能为NULL）
 */
WTSTickData* IndexFactory::sub_ticks(const char* fullCode)
{
	// 将合约代码加入订阅集合
	_subbed.insert(fullCode);
	// 解析完整代码，提取交易所和合约代码，例如："SHFE.rb2105" → {"SHFE", "rb2105"}
	auto ay = StrUtil::split(fullCode, ".");
	// 从DataManager获取该合约的当前Tick
	return _data_mgr->getCurTick(ay[1].c_str(), ay[0].c_str());
}
```

### 处理行情数据 handle_quote
接收成分合约的Tick行情，并分发给所有订阅了该合约的 IndexWorker。流程：
- 如果订阅集合 `_subbed` 中包含 newTick 对应的合约
  - 如果配置了线程池 `_pool`，将下述任务提交到线程池（否则直接同步执行）
    - `_workers` 中的每个指数工作器调用 handle_quote(newTick)，也就是检查自身是否订阅该合约，如果订阅了就进行指数计算
```cpp
/* @param newTick 新的Tick数据指针 */
void IndexFactory::handle_quote(WTSTickData* newTick)
```

### 推送指数Tick数据 push_tick
IndexWorker 计算出指数后调用，将指数作为一个新的Tick推送到系统
```cpp
/* @param newTick 新计算的指数Tick数据 */
void IndexFactory::push_tick(WTSTickData* newTick)
{
	// 写入DataManager
	// procFlag=1：仅写入，不更新缓存
	// 原因：指数是计算出来的，不需要缓存供查询
	// DataManager会将指数数据存储到文件并广播
	_data_mgr->writeTick(newTick, 1);
}
```

# 指数工作器 IndexWorker.h/cpp
负责单个指数的计算：订阅成分合约的行情，根据权重算法实时计算指数值。

## 成员
- `IndexFactory* _factor`：指数工厂指针
- `std::string _exchg`：指数所属交易所
- `std::string _code`：指数代码
- `std::string _trigger`：触发合约（或"time"）
- `uint32_t _timeout`：延时时间（毫秒），0=立即触发
- `uint64_t _recalc_time`：重算时间点（用于延时触发）
- `double _stand_scale`：标准化系数
- `WTSTickStruct _cache`：指数Tick缓存（用于计算开高低收）
- `WTSContractInfo*	_cInfo`：指数的合约信息
- `SpinMutex _mtx_data`：自旋锁（保护_weight_scales数据）
- `wt_hashmap<std::string, WeightFactor> _weight_scales`：成分合约映射表（代码 → 权重因子）
  - 其中 WeightFactor 包含：
    - `double _weight`：权重值
    - `WTSTickStruct _tick`：该成分的最新Tick
- `uint32_t	_weight_alg`：权重算法（0=固定，1=动态总持，2=动态成交量）
  - 0=固定：$$指数值 = \frac{{\sum {价格 \cdot 权重} }}{{总权重 \cdot 标准化系数}}$$
  - 1=动态总持：$$指数值 = \frac{{\sum {价格 \cdot 持仓量 \cdot 权重} }}{{\left( {\sum 持仓量 } \right) \cdot 总权重 \cdot 标准化系数}}$$
  - 2=动态成交量：$$指数值 = \frac{{\sum {价格 \cdot 成交量 \cdot 权重} }}{{\left( {\sum 成交量 } \right) \cdot 总权重 \cdot 标准化系数}}$$
- `StdThreadPtr	_thrd_trigger`：触发线程（延时触发时使用）
- `StdUniqueMutex	_mtx_trigger`：触发线程互斥锁
- `StdCondVariable	_cond_trigger`： 触发线程条件变量
- `bool _stopped`：停止标志
- `bool	_process`：处理标志（是否有待处理的触发）

## 方法

### 初始化 init
```cpp
bool IndexWorker::init(WTSVariant* config)
```
使用 config 来初始化当前指数工作器。配置参数 config 的例子：
```json
{
    "active": true,
    "exchg": "WTS",
    "code": "IMCI",
    "name": "工业原材料综合指数",
    "trigger": "SHFE.rb.HOT",
    "timeout": 50,
    "weight_alg": 1,
    "stand_scale": 100,
    "codes": [
        { "code": "SHFE.rb.HOT", "weight": 0.4 },
        { "code": "DCE.i.HOT", "weight": 0.35 },
        "CFFEX.IF2503", // 默认权重为 1.0
        { "code": "DCE.j", "weight": 0.25 }
    ]
}
```
- `active`：IndexFactory 的开关。只有当此项为 true 时，才会为这个指数创建 IndexWorker 实例。设为 false 可以临时禁用某个指数的计算，而无需删除其配置。
- `exchg`，`code`，`name`：定义了这个新生成的指数的交易所、代码和名称
- `trigger`
  - 值为合约例如 "SHFE.rb.HOT" 时，为指定合约触发模式，IndexWorker 只会在接收到该项记录的合约时，才启动一次计算流程
  - 值为时间时，为时间触发模式
- `timeout`：设置延时计算，单位为毫秒。
  - 当触发条件满足时，IndexWorker 并不会立即计算，而是会启动一个 timeout 毫秒的计时器
  - 只有当 timeout 毫秒计时结束后，才会执行一次计算，即使这中间可能有新的触发信号到来
  - 可以将一段时间内密集的多次触发合并为一次计算，极大地降低了在高频行情下的计算负载
- `weight_alg`：定义指数计算算法，0=固定，1=动态总持，2=动态成交量
- `stand_scale`：指数计算中的标准化系数
- `codes: [...]`、`commodities: [...]`：定义指数应包含哪些合约或品种，及其对应的权重
  - codes 的形式例如 `{ "code": "SHFE.rb.HOT", "weight": 0.4 }`，其中分别是合约代码和权重
  - commodities 的形式例如 `{ "code": "DCE.i", "weight": 1.5 }`，其中分别是品种代码和权重（该品种下的所有合约都使用该权重）
  - **如果 commodities 字段存在且不为空，则会优先处理它，codes 字段将被忽略。因此这两种方式是互斥的**

### 生成指数tick数据 generate_tick
```cpp
void IndexWorker::generate_tick()
```
根据所有成分的行情和权重算法计算指数值。

具体流程：
- 遍历 `_weight_scales` 中的所有成分合约，计算
  - 获取最大发生时间（日期+时间的毫秒格式）到 maxTime，最大交易日到 tDate
  - 累加当日成交量、当日成交额、当前总持仓量、总权重到 total_vol、total_amt、total_hold、total_weight
  - 根据权重算法 `_weight_alg`
    - 0=固定：total_base 为 1，累加 *最新价×权重* 到 total_value
    - 1=动态总持：累加当前总持仓量到 total_base，累加 *当前总持仓量×最新价×权重* 到 total_value
    - 2=动态成交量：累加当日成交量到 total_base，累加 *当日成交量×最新价×权重* 到 total_value
- 计算指数 $$index = \frac{{total\_value}}{{total\_base \cdot total\_weight \cdot \_stand\_scale}}$$
- 修正最大毫秒数 maxTime += `_timeout`，并获取其对应的日期和时间 tm32
- 更新 `_cache: WTSTickStruct`
  - index ——> _cache.price
  - max(_cache.high, index) ——> _cache.high
  - min(_cache.low, index) ——> _cache.low
  - tm32.date() ——> _cache.action_date
  - tm32.time_ms ——> _cache.action_time
  - total_vol ——> _cache.total_volume、total_hold ——> _cache.open_interest、total_amt ——> _cache.total_turnover
- 使用 `_cache` 创建一个Tick对象 newTick: *WTSTickData，并将合约信息 `_cInfo: WTSContractInfo*` 设置给它
- 指数工厂 `_factor: IndexFactory*` 推送即调用 push_tick(newTick)

### 处理成分合约的tick行情数据 handle_quote
```cpp
/* @param newTick 新的Tick数据指针 */
void IndexWorker::handle_quote(WTSTickData* newTick)
```
使用Tick数据 newTick 触发指数计算：
- 如果该Tick数据对应的合约不属于该指数的成分，直接返回
- 如果为指定合约触发模式但是该Tick数据对应的合约不是触发合约，直接返回
- （到下面说明是时间触发模式，或者指定合约触发模式且该Tick数据对应的合约就是触发合约）
  - 如果 `_timeout` 为零：调用 generate_tick() 立即生成指数 Tick
  - 否则：如果是第一次触发（`_thrd_trigger` 为空），则创建触发线程：
    - 当 `_stopped` 为假时
      - 当 `_process` 为假时（没有待处理的触发）：
        - `_mtx_trigger` 加锁
        - `_cond_trigger`.wait(`_mtx_trigger`)
      - （收到触发信号）等待：直到当前时间到达重算时间 `_recalc_time`
      - 调用 generate_tick() 生成指数 Tick，重置处理标志 `_process` 为 false
  - 如果 `_process` 为假
    - 设置 `_process` 为真，重新计算重算时间 `_recalc_time`，`_cond_trigger` 唤醒

# 交易时段状态监控器 StateMonitor.h/cpp

## 交易时段状态枚举 SimpleState
```cpp
typedef enum tagSimpleState
{
	SS_ORIGINAL,		// 未初始化状态（0）- 交易日开始前
	SS_INITIALIZED,		// 已初始化状态（1）- 系统就绪等待开盘
	SS_RECEIVING,		// 交易中状态（2）- 正在接收行情数据
	SS_PAUSED,			// 休息中状态（3）- 中途休盘时间
	SS_CLOSED,			// 已收盘状态（4）- 停止接收数据
	SS_PROCING,			// 收盘作业中状态（5）- 正在转储历史数据
	SS_PROCED,			// 盘后已处理状态（6）- 数据已归档
	SS_Holiday	= 99	// 节假日状态（99）- 非交易日
} SimpleState;
```

## 交易时段状态 StateInfo
```cpp
typedef struct _StateInfo
{
	char		_session[16];           // 交易时段标识符（如"TRADING"），最大15字符+'\0'
	uint32_t	_init_time;             // 初始化时间，格式HHMM（如0830表示8:30）
	uint32_t	_close_time;            // 收盘时间，格式HHMM（如1505表示15:05）
	uint32_t	_proc_time;             // 盘后处理时间，格式HHMM（如1530表示15:30）
	SimpleState	_state;                 // 当前状态（状态机的当前状态）
	WTSSessionInfo*	_sInfo;             // 交易时段详细信息指针（包含完整的时段配置）

	typedef struct _Section
	{
		uint32_t _from;                 // 区间开始时间，格式HHMM
		uint32_t _end;                  // 区间结束时间，格式HHMM
	} Section;
	
	std::vector<Section> _sections;     // 交易时间区间集合（支持多个不连续时段）
	
} StateInfo;
```

## 交易时段状态监控器 StateMonitor
用于管理多个交易时段的状态。

### 知识及理解

#### 品种的交易时段模板和节假日模板
总结：
- 任一品种包含`节假日模板`和`交易时段模板`（包含多个`交易时段`）
- 任一`交易时段`属于某个`交易日`，只有其对应的`交易日`不是节假日时，该`交易时段`才属于真正的可交易时间
- `交易时段模板`包含一个偏移
  - `== 0`：其包含的所有`交易时段`的`交易日`就是其所在`日历日`
  - `> 0`：交易时段的开始时间加上该偏移后
    - 小于 24 * 60 = 1440分钟：交易日是其所在日历日
    - 大于等于1440分钟：交易日是其所在日历日的下一天
  - `< 0`：交易日是所在日历日扣除该偏差（时差）后所在的日历日

例如：上海期货交易所（SHFE）的螺纹钢（rb）
- **节假日模板**: 假设2025年的元旦和春节假期安排如下：
  - 元旦: 1月1日（周三）为法定假日，休市。
  - 春节: 1月28日（周二，除夕前一天）晚上起至2月5日（周三）为法定假日，休市。2月6日（周四）恢复交易。
  - 周末: 所有周六、周日休市。
- **交易时段模板**: 螺纹钢的交易时间规则如下，这是一个典型的具有夜盘的品种。
  - 夜盘: 21:00 - 23:00
    - **夜盘时段属于下一个交易日**
    - **系统会为此模板配置一个正的偏移量**，例如 +480 分钟（8小时）
    - **如何判断是否为夜盘？加上偏移量后大于 24 * 60 = 1440 分钟**
  - 日盘 (上午): 09:00 - 10:15, 10:30 - 11:30
  - 日盘 (下午): 13:30 - 15:00
  - **关于偏移量**：
    - == 0；日内
- 结合**节假日模板**和**交易时段模板**推导出的详细交易时间表：
  * **12月31日, 星期二 (2024年)**
    * 日历日: 2024年12月31日
    * 交易日归属:
      * 白天的交易 (09:00 - 15:00) 属于 12月31日交易日。
      * **晚上的夜盘 (21:00 - 23:00) 本应属于下一个交易日。但由于下一个日历日（1月1日）是法定假日，所以今天晚上没有夜盘**。
    * 交易时间: 09:00 - 11:30, 13:30 - 15:00
  * **1月1日, 星期三 (元旦)**
    * 日历日: 2025年1月1日
    * 交易日归属: N/A
    * 交易时间: 全天休市
  * **1月2日, 星期四**
    * 日历日: 2025年1月2日
    * 交易日归属:
      * 白天的交易 (09:00 - 15:00) 属于 1月2日交易日。
      * 晚上的夜盘 (21:00 - 23:00) 属于 1月3日交易日。
    * 交易时间: 09:00 - 11:30, 13:30 - 15:00 以及 21:00 - 23:00
  * **1月3日, 星期五**
    * **日历日**: 2025年1月3日
    * **交易日归属**:
      * 白天的交易 (09:00 - 15:00) 属于 1月3日交易日
      * **晚上：由于下一个日历日（1月4日）是周六（非交易日），所以今天晚上没有夜盘**
    * 交易时间**: **09:00 - 11:30, 13:30 - 15:00
  * **1月4日 (周六) & 1月5日 (周日)**
    * 交易时间: 全天休市
    * 系统判断逻辑: 节假日模板生效，因为是周末

#### 状态变化
场景：郑州商品交易所的 PTA (精对苯二甲酸) 期货。这是一个典型的日盘品种，上午有两节交易，中间有15分钟的小节休息
- 交易时段模板：
  - 9:00-10:15：上午第一节
  - 10:30-11:30：上午第二节
  - 13:30-15:00：下午盘
- 偏移为 0：没有夜盘
- inittime: 08:30 (系统初始化时间)
- closetime: 15:05 (行情接收截止时间，比15:00收盘稍晚以接收最后数据)
- proctime: 15:30 (盘后数据处理开始时间)

一个完整交易日的状态变化：

**交易日：2025年10月10日 (星期五)**

| **时间点** | **当前状态** | **`run()` 方法中的关键判断** | **新状态** | **实际意义与系统行为** |
| :--- | :--- | :--- | :--- | :--- |
| **08:00:00** | `SS_ORIGINAL` | 当前时间 `0800` < `inittime` `0830`。 | `SS_ORIGINAL` | **系统休眠**。`StateMonitor` 处于待机状态，等待初始化时间的到来。此时不接收任何行情数据。 |
| **08:30:00** | `SS_ORIGINAL` | 当前时间 `0830` >= `inittime` `0830`。 | `SS_INITIALIZED` | **系统初始化**。系统被唤醒，进入盘前准备阶段。此时仍不接收行情数据。 |
| **08:59:00** | `SS_INITIALIZED` | 当前时间 `0859` < 第一个交易时段开始时间 `0900`。 | `SS_INITIALIZED` | **等待开盘**。系统已万事俱备，只等开盘信号。 |
| **09:00:00** | `SS_INITIALIZED` | 当前时间 `0900` >= 集合竞价/开盘时间，并且 `isInSections(0900)` 为 `true`。 | `SS_RECEIVING` | **上午第一节开盘**。状态切换到“接收中”。`DataManager` 的数据闸门正式打开，开始接收、存储和广播 `rb` 品种的实时行情。 |
| **10:00:00** | `SS_RECEIVING` | 当前时间 `1000` 仍在 `0900-1015` 区间内。 | `SS_RECEIVING` | **交易进行中**。系统持续接收行情数据。 |
| **10:15:00** | `SS_RECEIVING` | 当前时间 `1015` 不再处于任何交易区间内 (`!isInSections(1015)`)。 | `SS_PAUSED` | **上午小节休盘**。系统进入暂停状态。`DataManager` 停止接收新的行情数据。 |
| **10:25:00** | `SS_PAUSED` | 当前时间 `1025` 仍不处于任何交易区间内。 | `SS_PAUSED` | **小节休息中**。系统保持暂停，等待下一个交易时段的开始。 |
| **10:30:00** | `SS_PAUSED` | 当前时间 `1030` 处于 `1030-1130` 交易区间内 (`isInSections(1030)`)。 | `SS_RECEIVING` | **上午第二节开盘**。状态从“暂停”切换回“接收中”。`DataManager` 再次打开数据闸门。 |
| **11:30:00** | `SS_RECEIVING` | 当前时间 `1130` 不再处于任何交易区间内 (`!isInSections(1130)`)。 | `SS_PAUSED` | **午间休盘**。与上午小节休盘类似，系统再次进入暂停状态，停止接收数据，等待下午开盘。 |
| **12:30:00** | `SS_PAUSED` | 当前时间 `1230` 仍处于午休时段。 | `SS_PAUSED` | **午休进行中**。 |
| **13:30:00** | `SS_PAUSED` | 当前时间 `1330` 处于 `1330-1500` 交易区间内 (`isInSections(1330)`)。 | `SS_RECEIVING` | **下午盘开盘**。状态第三次切换到“接收中”，系统恢复行情接收。 |
| **15:00:00** | `SS_RECEIVING` | 当前时间 `1500` 仍处于 `1330-1500` 区间内（通常区间是左闭右开）。 | `SS_RECEIVING` | **下午收盘**。***但数据接收尚未停止，以确保能收到最后的结算价等收盘数据***。 |
| **15:05:00** | `SS_RECEIVING` | 当前时间 `1505` >= `closetime` `1505`。 | `SS_CLOSED` | **行情接收窗口关闭**。状态切换到“已收盘”。从此以后，`DataManager` 将拒绝所有新的行情数据。 |
| **15:20:00** | `SS_CLOSED` | 当前时间 `1520` < `proctime` `1530`。 | `SS_CLOSED` | **等待盘后处理**。系统处于静默状态，等待预设的数据归档时间。 |
| **15:30:00** | `SS_CLOSED` | 当前时间 `1530` >= `proctime` `1530`。 | `SS_PROCING` | **开始盘后数据处理**。状态切换到“处理中”。`StateMonitor` **立即触发** `_dt_mgr->transHisData(...)`，通知 `DataManager` 开始执行数据转储任务。 |
| **15:30:01** | `SS_PROCING` | 这是一个短暂的过渡状态。 | `SS_PROCED` | **盘后处理完成**。在下一个1秒的检查周期，状态立即切换为“已处理”，表示当天的所有工作已结束。 |
| **23:00:00** | `SS_PROCED` | 当前时间 `2300` > `inittime` `0830`。 | `SS_PROCED` | **当日工作已结束**。系统保持“已处理”状态，等待午夜的到来。 |
| **第二天 01:00:00** | `SS_PROCED` | 当前时间 `0100` 处于 `0000` 和 `inittime` `0830` 之间。 | `SS_ORIGINAL` | **状态重置，迎接新一天**。午夜过后，`StateMonitor` 检测到已进入新的日历日，且时间早于初始化时间，于是将状态重置为 `SS_ORIGINAL`，准备开始下一个完整的交易日生命周期。 |

### 成员

- `StateMap _map`：状态映射表，存储所有交易时段的状态信息
  - 本质上是 **map<交易时段ID, 交易时段状态StateInfo\*\>**
- `WTSBaseDataMgr* _bd_mgr`：基础数据管理器指针
- `DataManager* _dt_mgr`：数据管理器指针
- `StdThreadPtr _thrd`：监控线程智能指针，指向状态监控线程
  - 线程每秒检查一次状态并执行转换。
- `bool _stopped`：停止标志，控制监控线程的运行

### 方法

#### 初始化与生命周期

##### 初始化状态监控器 initialize
```cpp
/* @param filename 状态配置文件路径
 * @param bdMgr 基础数据管理器指针
 * @param dtMgr 数据管理器指针
 * @return bool 初始化成功返回true，失败返回false
 */
bool StateMonitor::initialize(const char* filename, WTSBaseDataMgr* bdMgr, DataManager* dtMgr)
```
配置JSON文件例如：
```json
{
  "TRADING": {			// 交易时段ID
    "inittime": 830,
    "closetime": 1505,
    "proctime": 1530
  },
  "NIGHT": {
    "inittime": 2030,
    "closetime": 2305,
    "proctime": 2330
  }
}
```
流程：
- 将参数 bdMgr 和 dtMgr 设置给基础数据管理器 `_bd_mgr` 和数据管理器 `_dt_mgr`
- 遍历每一个独立的交易时段ID
  - 创建一个新的 stateInfo: *StateInfo，在基础数据管理器 `_bd_mgr` 中查找对应交易时段ID的详细交易时间模板 ssInfo: WTSSessionInfo 
  - 读取并设置 stateInfo 的
    - _sInfo(WTSSessionInfo*)
	- _init_time: 初始化时间，在这个时间点，状态监控器会进入 SS_INITIALIZED 状态
    - _close_time: 收盘时间，这个时间点之后，状态会切换到 SS_CLOSED，停止接收行情数据
    - _proc_time: 盘后处理时间，在这个时间点，系统会开始进行数据转储等盘后作业，状态切换到 SS_PROCING
    - _session
    - 提取集合竞价区间、所有连续竞价区间到 _sections 中
  - 设置 `_map`：_map[stateInfo->_session] = stateInfo
  - 从基础数据管理器 `_bd_mgr` 中获取对应该交易时段ID的所有品种代码
    - 基于 ssInfo 的偏移时间设置 `_bd_mgr` 中这些品种对应节假日模板的当前交易日期（当前日期偏移）

##### 启动状态监控线程 run
创建并启动一个独立的监控线程，线程每秒检查一次时间和状态，并根据预定义的规则进行状态转换。
```cpp
void StateMonitor::run()
```
```mermaid
stateDiagram-v2
    [*] --> SS_ORIGINAL : 系统启动
    SS_ORIGINAL --> SS_INITIALIZED : 到达初始化时间
    SS_INITIALIZED --> SS_RECEIVING : 到达开盘时间
    SS_RECEIVING --> SS_PAUSED : 中途休盘
    SS_PAUSED --> SS_RECEIVING : 恢复交易
    SS_RECEIVING --> SS_CLOSED : 到达收盘时间
    SS_CLOSED --> SS_PROCING : 到达盘后处理时间
    SS_PROCING --> SS_PROCED : 处理完成
    SS_PROCED --> SS_ORIGINAL : 下一交易日
    
    SS_ORIGINAL --> SS_Holiday : 检测到节假日
    SS_INITIALIZED --> SS_Holiday : 检测到节假日
    SS_Holiday --> SS_ORIGINAL : 下一交易日
    
    note right of SS_RECEIVING : 可接收数据状态
    note right of SS_PROCING : 触发历史数据转储
```

流程：监控线程 `_thrd` 未创建时，以如下流程作为线程函数创建线程
- while(!`_stopped`)（持续运行直到触发停止信号）
  - 等待：与上次运行之后流程相距 1 秒
  - 检查停止标志 `_stopped`：触发则结束
  - 获取当前日期 curDate（YYYYMMDD）和时间戳 curMin（HHMM）
  - 遍历 `_map` 中的所有***交易时段***：
    - 获取该交易时段对应的详细配置 sInfo: WTSSessionInfo，并获取 curDate 的对应偏移日期 offDate
    - 根据 sInfo 的状态 `_state`
      - **未初始化 SS_ORIGINAL**
        - 如果：对应该 sInfo 对应的所有品种，当前时间都位于其节假日
          - _state 切换为 SS_Holiday
        - 否则如果：curMin >= sInfo的截止时间（**由于服务器维护、程序崩溃或其他原因，可能在当天的很晚才启动交易程序**）
          - _state 切换为 SS_CLOSED
        - 否则如果：curMin >= sInfo的第一个竞价时段的开始时间（**到达集合竞价时间**）
          - 如果 curMin 在sInfo的某交易区间内：_state 切换为 SS_RECEIVING
          - 否则：
            - 如果 curMin < sInfo的最后一个交易时段的结束时间：_state 切换为 SS_PAUSED
            - 否则：（**也就是大于等于最后一个交易时段的结束时间，并且小于截止时间**）_state 切换为 SS_RECEIVING
        - 否则如果：curMin >= sInfo的初始化时间
          - _state 切换为 SS_INITIALIZED
        - break
      - **已初始化 SS_INITIALIZED**
        - 如果：没有竞价时段或 curMin >= sInfo的第一个竞价时段的开始时间
          - 如果：curMin 不在交易区间内 && curMin < 最后一个交易时段的结束时间
            - _state 切换为 SS_PAUSED
          - 否则：（在交易区间内或大于等于最后一个交易时段的结束时间）
            - _state 切换为 SS_RECEIVING
        - break
      - **接收中 SS_RECEIVING**
        - 如果：curMin >= sInfo 的截止时间
          - _state 切换为 SS_CLOSED
        - 否则如果：curMin >= sInfo的第一个竞价时段的开始时间
          - 如果：curMin < sInfo的最后一个交易时段的结束时间
            - 如果：curMin 不在交易区间内
              - _state 切换为 SS_PAUSED
          - 否则：（大于等于最后一个交易时段的结束时间）
            - 保持
        - break
      - **暂停 SS_PAUSED**
        - 如果：对应该 sInfo 对应的所有品种，当前时间都位于其节假日
          - _state 切换为 SS_Holiday
        - 否则如果：curMin 在交易区间内
          - _state 切换为 SS_RECEIVING
        - break
      - **已收盘 SS_CLOSED**
        - 如果：curMin >= sInfo 的盘后处理时间
          - 如果：该交易时段还未完成盘后处理
            - _state 切换为 SS_PROCING
            - 对 `_dt_mgr` 触发历史数据存储
          - 否则：_state 切换为 SS_PROCED
        - 否则如果：(sInfo的第一个竞价时段的开始时间 <= curMin <= 最后一个交易时段的结束时间) &&  curMin 不在交易区间
          - _state 切换为 SS_PAUSED
        - break
      - **处理中 SS_PROCING**
        - _state 切换为 SS_PROCING（处理中是一个短暂的过渡状态，数据转储完成后立即转换为已处理状态）
        - break
      - **已处理 SS_PROCED**
      - **节假日 SS_Holiday**
        - 如果：(curMin < sInfo的初始化时间) && (对应该 sInfo 对应的所有品种，当前时间至少位于其中一个的交易日（非节假日）)
          - _state 切换为 SS_ORIGINAL
        - break
  - 如果：不是所有的时段都处于节假日状态 && 非节假日的时段都是处理中SS_PROCING状态
    - `_dt_mgr` 触发缓存清理（**所有非节假日的交易时段都已经完成了它们各自的数据接收工作，并都进入了盘后数据处理，这时触发一次全局的、最终的系统资源清理**）

##### 停止状态监控 stop
```cpp
void StateMonitor::stop()
{
	// 设置停止标志
	// 监控线程会在下次循环检查时发现并退出
	_stopped = true;

	// 等待线程结束
	// join()会阻塞当前线程，直到监控线程完全退出
	if (_thrd)
		_thrd->join();
}
```

#### 状态查询接口

##### 检查是否有任一时段处于指定状态 isAnyInState
```cpp
/* @param ss 要检查的状态
    * @return bool 至少有一个时段处于该状态返回true，否则返回false
    */
inline bool	isAnyInState(SimpleState ss) const
{
    auto it = _map.begin();
    for (; it != _map.end(); it++)
    {
        const StatePtr& sInfo = it->second;     // 获取StateInfo智能指针
        if (sInfo->_state == ss)                // 检查状态是否匹配
            return true;                        // 找到匹配的，立即返回true
    }

    return false;
}
```

##### 检查所有非SS_Holiday时段是否都处于指定状态 isAllInState
```cpp
/* @param ss 要检查的状态
 * @return bool 所有非节假日时段都处于该状态返回true，否则返回false
 */
inline bool	isAllInState(SimpleState ss) const
{
    auto it = _map.begin();
    for (; it != _map.end(); it++)
    {
        const StatePtr& sInfo = it->second;     // 获取StateInfo智能指针
        
        // 如果当前时段不是节假日，且状态不等于指定状态
        // 说明不是所有时段都处于指定状态
        if (sInfo->_state != SS_Holiday && sInfo->_state != ss)
            return false;                       // 找到不匹配的，返回false
    }

    return true;
}
```

##### 检查指定时段是否处于指定状态 isInState
```cpp
/* @param sid 交易时段标识符（如"TRADING"）
 * @param ss 要检查的状态
 * @return bool 时段存在且状态匹配返回true，否则返回false
 */
inline bool	isInState(const char* sid, SimpleState ss) const
{
    // 在哈希映射表中查找sid对应的StateInfo
    auto it = _map.find(sid);
    if (it == _map.end())                       // 如果找不到该时段
        return false;                           // 返回false

    // 获取StateInfo智能指针
    const StatePtr& sInfo = it->second;
    
    // 比较状态是否匹配
    return sInfo->_state == ss;
}
```

# 行情解析器适配器 ParserAdapter.h/cpp
```cpp
class ParserAdapter : public IParserSpi, private boost::noncopyable
```
参考 [Includes/note.ipynb/API 接口层/行情解析 IParserApi.h/行情解析器回调接口 IParserSpi](../Includes/note.ipynb)

## 行情解析器适配器类 ParserAdapter

### 成员
- `IParserApi* _parser_api`：Parser实例指针，指向实际的Parser对象（如ParserCTP、ParserXTP等）
  - 参考 [Includes/note.ipynb/API 接口层/行情解析 IParserApi.h/行情解析器接口 IParserApi](../Includes/note.ipynb)
- `FuncDeleteParser _remover`：Parser 销毁函数指针，用于销毁Parser实例
  - typedef void\(\*FuncDeleteParser\)\(wtp::IParserApi* &parser\);
- `WTSBaseDataMgr* _bd_mgr`：基础数据管理器指针
- `DataManager* _dt_mgr`：数据管理器指针，接收行情数据并进行存储和广播
- `IndexFactory* _idx_fact`：指数工厂指针，处理指数计算相关的逻辑
- `bool _stopped`：停止标志，控制适配器是否继续处理数据
- `ExchgFilter _exchg_filter`：交易所过滤器，存储允许订阅的交易所列表
  - typedef wt_hashset\<std::string\> `ExchgFilter`;
  - 例子：{"SHFE", "DCE"} 表示只允许订阅上期所和大商所
- `ExchgFilter _code_filter`：合约代码过滤器，存储允许订阅的合约代码列表
  - 支持格式："SHFE.rb2105" 即具体合约；或 "SHFE.rb" 即整个品种
  - _code_filter 优先级高于 _exchg_filter
- `WTSVariant* _cfg`：配置对象指针，保存初始化时的配置参数
- `std::string _id`：适配器唯一标识符

### 方法

#### 初始化与生命周期

##### 初始化 (从配置) init
```cpp
/* @param id 适配器ID
 * @param cfg 配置对象
 * @return bool 初始化成功返回true，失败返回false
 */
bool ParserAdapter::init(const char* id, WTSVariant* cfg)
```
参数 `cfg` 的例子：
```json
{
    "module": "ParserCTP",
    "filter": "SHFE,DCE",
    "code": "SHFE.rb.HOT,CFFEX.IF2503,DCE.i"
}
```
具体流程：
- 保存 id 和 cfg 到成员 `_id` 和 `_cfg` 中
- **（1）动态加载 Parser 模块**
  - 提取配置中的 `module` 字段，并根据 Windows/Linux 平台加上后缀 .dll/.so，查找并加载到 hInst，并从中
    - 获取 createParser 函数并运行，返回的 Paser 实例存储到 `_parser_api: IParserApi*`
    - 获取 deleteParser 函数指针到 `_remover`
- **（2）解析过滤器配置**：提取配置中 `filter`/`code` 字段（以逗号分隔）并保存到 `_exchg_filter`/`_code_filter`
- **（3）初始化 Paser 模块并订阅合约**：
  -  将 this 作为回调接口IParserSpi注册给 `_parser_api: IParserApi*`
  - 如果 `_code_filter` 非空
    - 遍历其所有代码并从 `_bd_mgr: WTSBaseDataMgr*` 中查找
  - 否则如果 `_exchg_filter` 非空
    - 遍历其所有交易所，并从 `_bd_mgr` 找到这些交易所的所有合约
  - 否则
    - 从 `_bd_mgr` 获取所有合约
  - `_parser_api` 订阅所有找到的合约

##### 初始化 (外部提供配置好的Parser实例) initExt
```cpp
/* @param id 适配器ID
 * @param api Parser实例指针
 * @return bool 初始化成功返回true，失败返回false
 */
bool ParserAdapter::initExt(const char* id, IParserApi* api)
```
具体流程：
- 保存 id 和 api 到成员 `_id` 和 `_parser_api: IParserApi*` 中
- 将 this 作为回调接口IParserSpi注册给 `_parser_api`

##### 启动运行 run
就是让 `_parser_api: IParserApi*` 连接服务器。
```cpp
/* @brief 启动Parser连接实现 */
bool ParserAdapter::run()
{
	if (_parser_api == NULL)
		return false;
	// 调用Parser的connect方法，启动连接
	// 连接是异步的，结果通过回调通知
	_parser_api->connect();
	return true;
}
```

#### IParserSpi 接口回调实现

##### 处理合约列表回调 handleSymbolList
```cpp
/* @param aySymbols 合约代码数组 */
void ParserAdapter::handleSymbolList( const WTSArray* aySymbols )
{
	// 当前为空实现：Parser推送的合约列表暂不处理
}
```

##### 处理Tick行情数据回调 handleQuote
```cpp
/**
 * @brief 处理Tick行情数据回调实现（最核心的方法）
 * 
 * 这是ParserAdapter最重要的方法，处理Parser推送的Tick行情数据。
 * 
 * 处理流程：
 * 1. 检查停止标志
 * 2. 验证数据有效性（日期不能为0）
 * 3. 获取或验证合约信息
 * 4. 写入DataManager（存储+广播）
 * 5. 转发给IndexFactory（指数计算）
 * 
 * 数据验证：
 * - actiondate：行情日期，格式YYYYMMDD
 * - tradingdate：交易日期，格式YYYYMMDD
 * - 任一为0表示数据无效，可能是Parser初始化阶段的数据
 * 
 * 合约信息处理：
 * - 优先使用数据对象中的合约信息（Parser可能已设置）
 * - 如果没有，从BaseDataMgr查询
 * - 设置到数据对象，供后续使用
 * 
 * @param quote Tick行情数据指针
 * @param procFlag 处理标志（0=正常，1=仅写入）
 */
void ParserAdapter::handleQuote( WTSTickData *quote, uint32_t procFlag )
{
	// 第一步：检查停止标志
	// 如果适配器已停止，丢弃所有数据
	if (_stopped)
		return;

	// 第二步：验证数据的基本有效性
	// 检查行情日期和交易日期是否有效
	// 日期为0通常表示：
	// 1. Parser初始化阶段的测试数据
	// 2. 数据解析错误
	// 3. 网络传输错误
	if (quote->actiondate() == 0 || quote->tradingdate() == 0)
		return;

	// 第三步：获取合约信息
	// 尝试从Tick对象获取合约信息（Parser可能已设置）
	WTSContractInfo* contract = quote->getContractInfo();
	if (contract == NULL)                       // 如果Tick中没有合约信息
	{
		// 从BaseDataMgr查询合约信息
		contract = _bd_mgr->getContract(quote->code(), quote->exchg());
		
		// 设置合约信息到Tick对象
		// 后续处理（DataManager、IndexFactory）可以直接使用
		// 避免重复查询，提高性能
		quote->setContractInfo(contract);
	}

	// 第四步：验证合约信息
	if (contract == NULL)                       // 如果合约信息不存在
		return;                                 // 丢弃数据（可能是无效合约或配置错误）

	// 第五步：写入DataManager
	// DataManager会：
	// 1. 检查是否可接收（canSessionReceive）
	// 2. 写入磁盘文件
	// 3. 更新内存缓存
	// 4. 广播到所有Caster
	if (!_dt_mgr->writeTick(quote, procFlag))
		return;                                 // 写入失败，不再继续处理

	// 第六步：转发给IndexFactory（指数计算）
	if (_idx_fact)                              // 如果指数工厂存在
		// IndexFactory会检查该Tick是否是指数成分
		// 如果是，触发指数重算
		_idx_fact->handle_quote(quote);
}
```

##### 处理委托队列数据回调 handleOrderQueue

##### 处理逐笔成交数据回调 handleTransaction

##### 处理逐笔委托数据回调 handleOrderDetail

##### 处理解析器日志回调 handleParserLog

##### 获取基础数据管理器 getBaseDataMgr

## 行情解析器适配器管理器类 ParserAdapterMgr

# 数据管理器 DataManager.h/cpp

## 成员
- `IDataWriter* _writer`：数据写入器接口指针，指向动态加载的数据存储模块实例，负责实际的数据持久化工作
- `FuncDeleteWriter	_remover`：Writer销毁函数指针，指向动态库中的deleteWriter函数，用于销毁Writer实例
  - typedef void\(*FuncDeleteWriter\)\(wtp::IDataWriter* &writer\);
- `WTSBaseDataMgr* _bd_mgr`：基础数据管理器指针
- `StateMonitor* _state_mon`：状态监控器指针，管理交易时段的状态
- `std::vector<IDataCaster*> _casters`：数据广播器集合，存储所有注册的数据广播器

好的，这是 `DataManager` 类的分层代码段，按照您要求的 Markdown 格式：

- **核心属性与构造函数**
  - 构造函数：`DataManager()`
  - 析构函数：`~DataManager()`

- **初始化与生命周期 (Initialization & Lifecycle)**
  - 初始化：`bool init(WTSVariant* params, WTSBaseDataMgr* bdMgr, StateMonitor* stMonitor)`
  - 释放资源：`void release()`
  - 添加数据广播器：`inline void add_caster(IDataCaster* caster)`
  - 添加扩展历史数据转储器：`void add_ext_dumper(const char* id, IHisDataDumper* dumper)`

- **核心数据写入接口 (Core Data Writing APIs)**
  - 写入Tick行情数据：`bool writeTick(WTSTickData* curTick, uint32_t procFlag)`
  - 写入委托队列数据：`bool writeOrderQueue(WTSOrdQueData* curOrdQue)`
  - 写入逐笔委托数据：`bool writeOrderDetail(WTSOrdDtlData* curOrdDetail)`
  - 写入逐笔成交数据：`bool writeTransaction(WTSTransData* curTrans)`

- **数据处理与查询 (Data Processing & Query)**
  - 触发历史数据转储：`void transHisData(const char* sid)`
  - 检查交易时段是否已处理完成：`bool isSessionProceeded(const char* sid)`
  - 获取合约的当前Tick数据：`WTSTickData* getCurTick(const char* code, const char* exchg = "")`

- **IDataWriterSink 接口回调实现 (IDataWriterSink Interface Implementation)**
  - 获取基础数据管理器：`virtual IBaseDataMgr* getBDMgr() override`
  - 检查交易时段是否可以接收数据：`virtual bool canSessionReceive(const char* sid) override`
  - 广播Tick数据：`virtual void broadcastTick(WTSTickData* curTick) override`
  - 广播委托队列数据：`virtual void broadcastOrdQue(WTSOrdQueData* curOrdQue) override`
  - 广播逐笔委托数据：`virtual void broadcastOrdDtl(WTSOrdDtlData* curOrdDtl) override`
  - 广播逐笔成交数据：`virtual void broadcastTrans(WTSTransData* curTrans) override`
  - 获取交易时段对应的品种集合：`virtual CodeSet* getSessionComms(const char* sid) override`
  - 获取品种的交易日：`virtual uint32_t getTradingDate(const char* pid) override`
  - 输出日志信息：`virtual void outputLog(WTSLogLevel ll, const char* message) override`

## 方法

### 初始化与生命周期

#### 初始化 init

#### 添加数据广播器 add_caster

#### 添加扩展历史数据转储器 add_ext_dumper

### 核心数据写入接口

#### 写入Tick行情数据 writeTick

#### 写入委托队列数据 writeOrderQueue

#### 写入逐笔委托数据 writeOrderDetail

#### 写入逐笔成交数据 writeTransaction

### 数据处理与查询

#### 触发历史数据转储 transHisData

#### 检查交易时段是否已处理完成 isSessionProceeded

#### 获取合约的当前Tick数据 getCurTick

### IDataWriterSink 接口回调实现

#### 获取基础数据管理器 getBDMgr

#### 检查交易时段是否可以接收数据 canSessionReceive

#### 广播Tick数据 broadcastTick

#### 广播委托队列数据 broadcastOrdQue

#### 广播逐笔委托数据 broadcastOrdDtl

#### 广播逐笔成交数据 broadcastTrans

#### 获取交易时段对应的品种集合 getSessionComms

#### 获取品种的交易日 getTradingDate

#### 输出日志信息 outputLog